# GPT-4 Zero-Shot — Mental Health EDS

Reproducible zero-shot evaluation from *Zero-Shot vs. Fine-Tuned* (structured prompts, constraint-based parsing, temperature=0). Requires `OPENAI_API_KEY`.

Legacy T5 zero-shot notebooks are in `archieve/`.


In [1]:
import os
import sys
from dotenv import load_dotenv
load_dotenv()  # automatically finds .env in parent dirs
PROJECT_ROOT = os.getcwd()
if not os.path.isdir(os.path.join(PROJECT_ROOT, 'zeroshot')):
    PROJECT_ROOT = os.path.abspath(os.path.join(PROJECT_ROOT, '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.environ.setdefault('WANDB_DISABLED', 'true')


'true'

In [2]:
from zeroshot.classifier import ZeroShotClassifier
from zeroshot.evaluate import evaluate_zero_shot, load_split_df, resolve_base_dir
from zeroshot.labels import DATASET_CONFIGS

CONFIG_KEY = 'mental_health'
BASE_DIR = resolve_base_dir()
MAX_SAMPLES = int(os.getenv('ZEROSHOT_MAX_SAMPLES', '0')) or None  # cap for dry runs

# gpt-4o-mini is cheaper for development; use gpt-4 for paper replication
MODEL = os.getenv('ZEROSHOT_MODEL', 'gpt-4o-mini')
BACKEND = os.getenv('ZEROSHOT_BACKEND', 'openai')  # or mistral_local

classifier = ZeroShotClassifier(model=MODEL, backend=BACKEND, temperature=0.0, max_tokens=10)
config = DATASET_CONFIGS[CONFIG_KEY]
print(f'Dataset: {config.name} | Model: {MODEL} | Base: {BASE_DIR}')


Dataset: Mental Health EDS | Model: gpt-4.1-mini | Base: /Users/kirthi/Documents/UCBerkeley/kirthi_portfolio/MIDS/Academic_Projects/Medical_NLP_Zeroshot_vs_Finetune_v2/


In [3]:
val_df = load_split_df(config, 'validation', BASE_DIR)
test_df = load_split_df(config, 'test', BASE_DIR)
print(f'Validation: {len(val_df)} | Test: {len(test_df)}')
print(val_df[config.label_column].value_counts().head())


Validation: 9624 | Test: 9622
status_combined
Depressive_Spectrum    2407
Anxiety/Stress         2406
Normal                 2406
Bipolar/Personality    2405
Name: count, dtype: int64


In [4]:
val_results = evaluate_zero_shot(
    val_df,
    CONFIG_KEY,
    classifier,
    dataset_name='validation',
    max_samples=MAX_SAMPLES,
    results_dir=os.path.join(PROJECT_ROOT, 'Results'),
)


Zero-shot inference: 100%|██████████| 9624/9624 [13:25:27<00:00,  5.02s/it]       



ZERO-SHOT EVALUATION — MENTAL HEALTH EDS (validation)
Model: gpt-4.1-mini | Backend: openai
Valid pairs: 9624/9624
Accuracy:     0.7416
F1 (macro):   0.7354
F1 (weighted): 0.7354
Precision:    0.7630
Recall:       0.7416

Classification report:
                     precision    recall  f1-score   support

DEPRESSIVE_SPECTRUM       0.71      0.81      0.76      2407
     ANXIETY_STRESS       0.71      0.76      0.73      2406
BIPOLAR_PERSONALITY       0.91      0.51      0.66      2405
             NORMAL       0.72      0.88      0.79      2406

           accuracy                           0.74      9624
          macro avg       0.76      0.74      0.74      9624
       weighted avg       0.76      0.74      0.74      9624


Saved confusion matrix: /Users/kirthi/Documents/UCBerkeley/kirthi_portfolio/MIDS/Academic_Projects/Medical_NLP_Zeroshot_vs_Finetune_v2/Results/zeroshot_confusion_matrix_mental_health_validation_openai.png
Saved predictions: /Users/kirthi/Documents/UCBerkeley/kir

In [5]:
test_results = evaluate_zero_shot(
    test_df,
    CONFIG_KEY,
    classifier,
    dataset_name='test',
    max_samples=MAX_SAMPLES,
    results_dir=os.path.join(PROJECT_ROOT, 'Results'),
)


Zero-shot inference: 100%|██████████| 9622/9622 [1:45:37<00:00,  1.52it/s]  



ZERO-SHOT EVALUATION — MENTAL HEALTH EDS (test)
Model: gpt-4.1-mini | Backend: openai
Valid pairs: 9622/9622
Accuracy:     0.7428
F1 (macro):   0.7364
F1 (weighted): 0.7364
Precision:    0.7639
Recall:       0.7428

Classification report:
                     precision    recall  f1-score   support

DEPRESSIVE_SPECTRUM       0.72      0.82      0.77      2406
     ANXIETY_STRESS       0.72      0.74      0.73      2405
BIPOLAR_PERSONALITY       0.91      0.52      0.66      2405
             NORMAL       0.71      0.89      0.79      2406

           accuracy                           0.74      9622
          macro avg       0.76      0.74      0.74      9622
       weighted avg       0.76      0.74      0.74      9622


Saved confusion matrix: /Users/kirthi/Documents/UCBerkeley/kirthi_portfolio/MIDS/Academic_Projects/Medical_NLP_Zeroshot_vs_Finetune_v2/Results/zeroshot_confusion_matrix_mental_health_test_openai.png
Saved predictions: /Users/kirthi/Documents/UCBerkeley/kirthi_portfoli

## Optional: Mistral-7B-Instruct (local)

Set `ZEROSHOT_BACKEND=mistral_local` and run on a GPU. Uses 4-bit quantization by default.
